# Conjunto de datos completo sin clusterización

In [2]:
#Importaciones
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

#Lectura de datos
datos = pd.read_excel('03_Clusterizacion_CTNET.xlsx')
datos.head(24)

,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM
0,2022-09-01 00:00:00,0.000000,19,77,0,0,Noche,Noche
1,2022-09-01 01:00:00,0.000000,19,82,0,1,Noche,Noche
2,2022-09-01 02:00:00,0.000000,18,85,0,2,Noche,Noche
3,2022-09-01 03:00:00,0.000000,18,87,0,3,Noche,Noche
4,2022-09-01 04:00:00,0.000000,18,88,0,4,Noche,Noche
5,2022-09-01 05:00:00,0.000000,17,86,0,5,Noche,Noche
6,2022-09-01 06:00:00,0.000000,18,89,0,6,Soleado,Lluvioso
7,2022-09-01 07:00:00,6.584959,18,95,0,7,Soleado,Lluvioso
8,2022-09-01 08:00:00,560.422022,18,100,0,8,Soleado,Lluvioso
9,2022-09-01 09:00:00,7720.582326,18,100,1,9,Soleado,Lluvioso


In [3]:
datos["Generacion_prev_hour"] = datos["Generación"].shift(1)
datos["Generacion_prev_day"] = datos["Generación"].shift(24)
datos = datos.dropna(how="any", axis= 0)

Definimos X y y

In [4]:
datos_dia = datos.copy()
datos_dia.head(10)

,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day
24,2022-09-02 00:00:00,0.000000,19,76,0,0,Noche,Noche,0.000000,0.000000
25,2022-09-02 01:00:00,0.000000,18,81,0,1,Noche,Noche,0.000000,0.000000
26,2022-09-02 02:00:00,0.000000,18,84,0,2,Noche,Noche,0.000000,0.000000
27,2022-09-02 03:00:00,0.000000,18,86,0,3,Noche,Noche,0.000000,0.000000
28,2022-09-02 04:00:00,0.000000,17,86,0,4,Noche,Noche,0.000000,0.000000
29,2022-09-02 05:00:00,0.000000,17,88,0,5,Noche,Noche,0.000000,0.000000
30,2022-09-02 06:00:00,0.000000,17,91,0,6,Soleado,Lluvioso,0.000000,0.000000
31,2022-09-02 07:00:00,0.000000,17,94,0,7,Soleado,Lluvioso,0.000000,6.584959
32,2022-09-02 08:00:00,438.814997,16,97,0,8,Soleado,Lluvioso,0.000000,560.422022
33,2022-09-02 09:00:00,5908.000884,17,93,1,9,Soleado,Lluvioso,438.814997,7720.582326


In [5]:
columns = datos_dia.drop(columns=["Fecha", "Generación", "Cluster KMeans", "Cluster GMM"]).columns

In [6]:
X = datos_dia[columns]
X

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
24,19,76,0,0,0.0,0.0
25,18,81,0,1,0.0,0.0
26,18,84,0,2,0.0,0.0
27,18,86,0,3,0.0,0.0
28,17,86,0,4,0.0,0.0
...,...,...,...,...,...,...
18285,22,45,0,20,1450.0,0.0
18286,20,54,0,21,0.0,0.0
18287,18,62,0,22,0.0,0.0
18288,17,69,0,23,0.0,0.0


In [7]:
y = datos_dia[['Generación']]
y

,Generación
24,0.0
25,0.0
26,0.0
27,0.0
28,0.0
...,...
18285,0.0
18286,0.0
18287,0.0
18288,0.0


Dividimos entrenamiento, validación y prueba

In [8]:
train_size = int(0.7 * len(X))
val_size = int(0.85 * len(X))

In [9]:
# Entrenamiento, validación y prueba, 75, 15 y 15
X_train, y_train =  X.iloc[:train_size, :], y.iloc[:train_size, :]
X_val, y_val = X.iloc[train_size:val_size, :], y.iloc[train_size:val_size, :]
X_test, y_test = X.iloc[val_size:, :],  y.iloc[val_size:,:]

print(f'X_train: {len(X_train)}, y_train: {len(y_train)}')
print(f'X_val: {len(X_val)}, y_val: {len(y_val)}')
print(f'X_test: {len(X_test)}, y_test: {len(y_test)}')

X_train: 12786, y_train: 12786
X_val: 2740, y_val: 2740
X_test: 2740, y_test: 2740


## Escalar con MinMaxScaler

In [10]:
from sklearn.preprocessing import MinMaxScaler

In [11]:
x_scaler = MinMaxScaler().fit(X_train)
x_scaler

MinMaxScaler()

In [12]:
X_train_scaled = x_scaler.transform(X_train)
print(X_train_scaled)
print(X_train_scaled.shape)

[[0.5        0.74736842 0.         0.         0.         0.        ]
 [0.47368421 0.8        0.         0.04347826 0.         0.        ]
 [0.47368421 0.83157895 0.         0.08695652 0.         0.        ]
 ...
 [0.36842105 0.77894737 0.07142857 0.60869565 0.71913333 0.34156667]
 [0.36842105 0.75789474 0.07142857 0.65217391 0.70556667 0.33793333]
 [0.36842105 0.72631579 0.07142857 0.69565217 0.79483333 0.24856667]]
(12786, 6)


In [13]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_train_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
24,0.500000,0.747368,0.000000,0.000000,0.000000,0.000000
25,0.473684,0.800000,0.000000,0.043478,0.000000,0.000000
26,0.473684,0.831579,0.000000,0.086957,0.000000,0.000000
27,0.473684,0.852632,0.000000,0.130435,0.000000,0.000000
28,0.447368,0.852632,0.000000,0.173913,0.000000,0.000000
...,...,...,...,...,...,...
12805,0.342105,0.736842,0.142857,0.521739,0.843733,0.344767
12806,0.394737,0.652632,0.142857,0.565217,0.829000,0.338200
12807,0.368421,0.778947,0.071429,0.608696,0.719133,0.341567
12808,0.368421,0.757895,0.071429,0.652174,0.705567,0.337933


In [14]:
X_val_scaled = x_scaler.transform(X_val)
print(X_val_scaled)
print(X_val_scaled.shape)

[[0.34210526 0.72631579 0.         0.73913043 0.70433333 0.24723333]
 [0.34210526 0.69473684 0.         0.7826087  0.702      0.2108    ]
 [0.34210526 0.69473684 0.         0.82608696 0.4138     0.01416667]
 ...
 [0.81578947 0.25263158 0.14285714 0.7826087  0.9109     0.82416667]
 [0.78947368 0.30526316 0.07142857 0.82608696 0.82416667 0.45596667]
 [0.73684211 0.35789474 0.         0.86956522 0.46683333 0.0394    ]]
(2740, 6)


In [15]:
X_val_scaled_df = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
X_val_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
12810,0.342105,0.726316,0.000000,0.739130,0.704333,0.247233
12811,0.342105,0.694737,0.000000,0.782609,0.702000,0.210800
12812,0.342105,0.694737,0.000000,0.826087,0.413800,0.014167
12813,0.315789,0.726316,0.000000,0.869565,0.043467,0.000000
12814,0.289474,0.768421,0.000000,0.913043,0.000000,0.000000
...,...,...,...,...,...,...
15545,0.894737,0.178947,0.357143,0.695652,0.959467,0.935400
15546,0.868421,0.200000,0.214286,0.739130,0.940767,0.910900
15547,0.815789,0.252632,0.142857,0.782609,0.910900,0.824167
15548,0.789474,0.305263,0.071429,0.826087,0.824167,0.455967


In [16]:
X_test_scaled = x_scaler.transform(X_test)
print(X_test_scaled)
print(X_test_scaled.shape)

[[0.68421053 0.42105263 0.         0.91304348 0.0379     0.        ]
 [0.65789474 0.49473684 0.         0.95652174 0.         0.        ]
 [0.60526316 0.56842105 0.         1.         0.         0.        ]
 ...
 [0.47368421 0.6        0.         0.95652174 0.         0.        ]
 [0.44736842 0.67368421 0.         1.         0.         0.        ]
 [0.42105263 0.71578947 0.         0.         0.         0.        ]]
(2740, 6)


In [17]:
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
X_test_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
15550,0.684211,0.421053,0.0,0.913043,0.037900,0.0
15551,0.657895,0.494737,0.0,0.956522,0.000000,0.0
15552,0.605263,0.568421,0.0,1.000000,0.000000,0.0
15553,0.578947,0.600000,0.0,0.000000,0.000000,0.0
15554,0.578947,0.631579,0.0,0.043478,0.000000,0.0
...,...,...,...,...,...,...
18285,0.578947,0.421053,0.0,0.869565,0.048333,0.0
18286,0.526316,0.515789,0.0,0.913043,0.000000,0.0
18287,0.473684,0.600000,0.0,0.956522,0.000000,0.0
18288,0.447368,0.673684,0.0,1.000000,0.000000,0.0


In [18]:
x_scaller_all = MinMaxScaler().fit(X)
print(x_scaller_all)

MinMaxScaler()


In [19]:
X_scaled = x_scaller_all.transform(X)
print(X_scaled)
print(X_scaled.shape)

[[0.48717949 0.75257732 0.         0.         0.         0.        ]
 [0.46153846 0.80412371 0.         0.04347826 0.         0.        ]
 [0.46153846 0.83505155 0.         0.08695652 0.         0.        ]
 ...
 [0.46153846 0.60824742 0.         0.95652174 0.         0.        ]
 [0.43589744 0.68041237 0.         1.         0.         0.        ]
 [0.41025641 0.72164948 0.         0.         0.         0.        ]]
(18266, 6)


In [20]:
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
X_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
24,0.487179,0.752577,0.0,0.000000,0.000000,0.0
25,0.461538,0.804124,0.0,0.043478,0.000000,0.0
26,0.461538,0.835052,0.0,0.086957,0.000000,0.0
27,0.461538,0.855670,0.0,0.130435,0.000000,0.0
28,0.435897,0.855670,0.0,0.173913,0.000000,0.0
...,...,...,...,...,...,...
18285,0.564103,0.432990,0.0,0.869565,0.048333,0.0
18286,0.512821,0.525773,0.0,0.913043,0.000000,0.0
18287,0.461538,0.608247,0.0,0.956522,0.000000,0.0
18288,0.435897,0.680412,0.0,1.000000,0.000000,0.0


In [21]:
y_scaler = MinMaxScaler().fit(y_train)
print(y_scaler)

MinMaxScaler()


In [22]:
y_train_scaled = y_scaler.transform(y_train)
print(y_train_scaled)
print(y_train_scaled.shape)

[[0.        ]
 [0.        ]
 [0.        ]
 ...
 [0.70556667]
 [0.79483333]
 [0.70433333]]
(12786, 1)


In [23]:
y_train_scaled_df = pd.DataFrame(y_train_scaled, index=y_train.index, columns=y_train.columns)
y_train_scaled_df

,Generación
24,0.000000
25,0.000000
26,0.000000
27,0.000000
28,0.000000
...,...
12805,0.829000
12806,0.719133
12807,0.705567
12808,0.794833


In [24]:
y_val_scaled = y_scaler.transform(y_val)
print(y_val_scaled)
print(y_val_scaled.shape)

[[0.702     ]
 [0.4138    ]
 [0.04346667]
 ...
 [0.82416667]
 [0.46683333]
 [0.0379    ]]
(2740, 1)


In [25]:
y_val_scaled_df = pd.DataFrame(y_val_scaled, index=y_val.index, columns=y_val.columns)
y_val_scaled_df

,Generación
12810,0.702000
12811,0.413800
12812,0.043467
12813,0.000000
12814,0.000000
...,...
15545,0.940767
15546,0.910900
15547,0.824167
15548,0.466833


In [26]:
y_test_scaled = y_scaler.transform(y_test)
print(y_test_scaled)
print(y_test_scaled.shape)

[[0.]
 [0.]
 [0.]
 ...
 [0.]
 [0.]
 [0.]]
(2740, 1)


In [27]:
y_test_scaled_df = pd.DataFrame(y_test_scaled, index=y_test.index, columns=y_test.columns)
y_test_scaled_df

,Generación
15550,0.0
15551,0.0
15552,0.0
15553,0.0
15554,0.0
...,...
18285,0.0
18286,0.0
18287,0.0
18288,0.0


In [28]:
y_scaller_all = MinMaxScaler().fit(y)
print(y_scaller_all)

MinMaxScaler()


In [29]:
y_scaled = y_scaller_all.transform(y)
print(y_scaled)
print(y_scaled.shape)

[[0.]
 [0.]
 [0.]
 ...
 [0.]
 [0.]
 [0.]]
(18266, 1)


In [30]:
y_scaled_df = pd.DataFrame(y_scaled, index=y.index, columns=y.columns)
y_scaled_df

,Generación
24,0.0
25,0.0
26,0.0
27,0.0
28,0.0
...,...
18285,0.0
18286,0.0
18287,0.0
18288,0.0


## Definición de modelos

### RandomForest

In [31]:
from lightgbm import LGBMRegressor
import optuna
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import seaborn as sns
from sklearn.metrics import mean_absolute_percentage_error as mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error as mean_absolute_error
from sklearn.metrics import mean_squared_error as mean_squared_error
from sklearn.metrics import r2_score as r2_score

In [32]:
# Inicializar listas para métricas
LightGBM_model = LGBMRegressor(num_leaves=500, subsample= 0.10698460631792395, colsample_bytree= 0.7272836809565294, min_data_in_leaf= 85)
LightGBM_model.fit(X_train_scaled_df, y_train_scaled_df)
resultados = pd.DataFrame(index = y_test_scaled_df.index, columns=["LightGBM"])
#Ciclo diario de predicción
for i in range(len(X_test)):
    inicio = i * 1
    fin = inicio + 1

    X_test_seg = X_test_scaled_df.iloc[inicio:fin, :]
    y_test_seg = y_test_scaled_df.iloc[inicio:fin]

    if len(X_test_seg) < 1:
        break

    y_pred = LightGBM_model.predict(X_test_seg)
    y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1))
    y_pred = np.clip(y_pred, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

    resultados.iloc[i, 0] = y_pred[0, 0]

[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.085401 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 684
[LightGBM] [Info] Number of data points in the train set: 12786, number of used features: 6
[LightGBM] [Info] Start training from score 0.296336
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spli

In [33]:
resultados

,LightGBM
15550,0.0
15551,0.0
15552,0.0
15553,0.0
15554,0.0
...,...
18285,98.179402
18286,0.0
18287,0.0
18288,4.376568


In [34]:
predicciones = y_test.copy()
predicciones

,Generación
15550,0.0
15551,0.0
15552,0.0
15553,0.0
15554,0.0
...,...
18285,0.0
18286,0.0
18287,0.0
18288,0.0


In [35]:
predicciones["LightGBM"] = resultados["LightGBM"]
predicciones

,Generación,LightGBM
15550,0.0,0.0
15551,0.0,0.0
15552,0.0,0.0
15553,0.0,0.0
15554,0.0,0.0
...,...,...
18285,0.0,98.179402
18286,0.0,0.0
18287,0.0,0.0
18288,0.0,4.376568


In [36]:
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['LightGBM'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['LightGBM']):.4f}")

MAE: 926.3719
RMSE: 2052.1413
R²: 0.9686


## Random Forest

In [37]:
from sklearn.ensemble import RandomForestRegressor

In [38]:
#Modelo LightGBM
RF_model = RandomForestRegressor(
    criterion="squared_error",
    random_state=0,
    n_estimators=400,
    min_impurity_decrease=0,
    max_depth=None,
    bootstrap=True
)
RF_model.fit(X_train_scaled_df, y_train_scaled_df)
# Inicializar listas para métricas
resultados = pd.DataFrame(index = y_test_scaled_df.index, columns=["Random Forest"])
#Ciclo diario de predicción
for i in range(len(X_test)):
    inicio = i * 1
    fin = inicio + 1

    X_test_seg = X_test_scaled_df.iloc[inicio:fin, :]
    y_test_seg = y_test_scaled_df.iloc[inicio:fin]

    if len(X_test_seg) < 1:
        break

    y_pred = RF_model.predict(X_test_seg)
    y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1))
    y_pred = np.clip(y_pred, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

    resultados.iloc[i, 0] = y_pred[0, 0]

In [39]:
predicciones["Random Forest"] = resultados["Random Forest"]
predicciones

,Generación,LightGBM,Random Forest
15550,0.0,0.0,0.0
15551,0.0,0.0,0.0
15552,0.0,0.0,0.0
15553,0.0,0.0,0.0
15554,0.0,0.0,0.0
...,...,...,...
18285,0.0,98.179402,0.0
18286,0.0,0.0,0.0
18287,0.0,0.0,0.0
18288,0.0,4.376568,0.0


## Preparación redes neuronales

In [40]:
import numpy as np
import pandas as pd

def create_sliding_window_with_index(data_X, data_y, lookback):
    X, y, indices = [], [], []
    
    # Asegurar que `data_y` tiene los mismos índices que `data_X`
    data_y = data_y.reindex(data_X.index)

    max_index = len(data_X) - lookback

    for i in range(max_index):
        X.append(data_X.iloc[i:i + lookback].values)  # Ventana de entrada
        
        # Obtener el índice correcto en `data_y`
        y_index = data_X.index[i + lookback]

        # Extraer el valor correspondiente de `data_y`
        if y_index in data_y.index:
            y_value = data_y.loc[y_index]
            if isinstance(y_value, pd.Series):  # Si devuelve una serie, extraer el valor
                y_value = y_value.iloc[0]
        else:
            y_value = np.nan  # Si no está, asignamos NaN

        y.append(y_value)
        indices.append(y_index)  # 🔹 Guardamos el índice original de `data_y`

    # Convertimos `X` en un array y `y` en DataFrame conservando sus índices originales
    X_array = np.array(X)
    y_df = pd.DataFrame(y, index=indices, columns=['y'])  # 🔹 Conservamos los índices originales

    return X_array, y_df


In [41]:
lookback = 48  # Puedes ajustar a 24, 72, etc.

# Aplicar la ventana deslizante a cada conjunto
X_train_windowed, y_train_windowed = create_sliding_window_with_index(X_train_scaled_df, y_train_scaled_df, lookback)
X_val_windowed, y_val_windowed = create_sliding_window_with_index(X_val_scaled_df, y_val_scaled_df, lookback)
X_test_windowed, y_test_windowed = create_sliding_window_with_index(X_test_scaled_df, y_test_scaled_df, lookback)


In [42]:
print(f'X_train: {X_train_windowed.shape}, y_train: {y_train_windowed.shape}')
print(f'X_val: {X_val_windowed.shape}, y_val: {y_val_windowed.shape}')
print(f'X_test: {X_test_windowed.shape}, y_test: {y_test_windowed.shape}')

X_train: (12738, 48, 6), y_train: (12738, 1)
X_val: (2692, 48, 6), y_val: (2692, 1)
X_test: (2692, 48, 6), y_test: (2692, 1)


## CTNET

In [43]:
import tensorflow as tf
from tensorflow.keras import layers

In [44]:
def compile_and_fit(model, xtrain=X_train_windowed, ytrain=y_train_windowed, learning_rate=0.0001):
    model.compile(loss=[tf.keras.losses.MeanSquaredError()],
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=[tf.keras.metrics.RootMeanSquaredError(), tf.keras.metrics.MeanAbsolutePercentageError(), tf.keras.metrics.MeanAbsoluteError()])
    
    history = model.fit(xtrain, ytrain, epochs=50,
                        batch_size=512, validation_split=0.2, verbose=1)
    return history

def Loss(train_loss, valid_loss):
    plt.plot(train_loss)
    plt.plot(valid_loss)
    plt.rcParams["figure.figsize"] = (15, 3)
    plt.title('Model Losses')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train Loss', 'Validation Loss'], loc='upper left')
    plt.savefig('out/loss_plot.png')
    plt.show()

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization()(inputs)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=128, kernel_size=2, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(norm_x, norm_x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    return norm_x

def build_model(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout=0, mlp_dropout=0):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs
    
    for _ in range(num_transformer_blocks):
        enc_out = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)
    
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(enc_out, enc_out)
    res = x + enc_out
    x = layers.LayerNormalization(epsilon=1e-6)(res)
    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)
    x = layers.Dense(832, activation="relu")(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(mlp_dropout)(x)

    outputs = layers.Dense(1)(x)
    
    return tf.keras.Model(inputs, outputs)

In [45]:
CTNET = build_model((X_train_windowed.shape[1], X_train_windowed.shape[2]), head_size=4, num_heads=3, ff_dim=32, num_transformer_blocks=3, mlp_units=[256], mlp_dropout=0.3, dropout=0.2)

In [46]:
history = compile_and_fit(CTNET)

Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 310s 3s/step - loss: 0.2354 - mean_absolute_error: 0.3082 - mean_absolute_percentage_error: 4403025.0000 - root_mean_squared_error: 0.4851 - val_loss: 0.1420 - val_mean_absolute_error: 0.2488 - val_mean_absolute_percentage_error: 25142792.0000 - val_root_mean_squared_error: 0.3768
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 49s 2s/step - loss: 0.2061 - mean_absolute_error: 0.3110 - mean_absolute_percentage_error: 31955324.0000 - root_mean_squared_error: 0.4539 - val_loss: 0.1124 - val_mean_absolute_error: 0.2677 - val_mean_absolute_percentage_error: 78838264.0000 - val_root_mean_squared_error: 0.3352
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 66s 3s/step - loss: 0.1618 - mean_absolute_error: 0.3236 - mean_absolute_percentage_error: 87348312.0000 - root_mean_squared_error: 0.4022 - val_loss: 0.1087 - val_mean_absolute_error: 0.3082 - val_mean_absolute_percentage_error: 161456448.0000 - val_root_mean_squared_error: 0.3296
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 8

In [47]:
CTNET_predictions = CTNET.predict(X_test_windowed)
CTNET_predictions

85/85 ━━━━━━━━━━━━━━━━━━━━ 38s 258ms/step


array([[0.00023607],
       [0.00088889],
       [0.00051992],
       ...,
       [0.00254314],
       [0.00254289],
       [0.00224794]], dtype=float32)

In [48]:
CTNET_predictions = y_scaler.inverse_transform(CTNET_predictions.reshape(-1, 1))
CTNET_predictions = np.clip(CTNET_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [49]:
resultados = pd.DataFrame(CTNET_predictions, index = y_test_windowed.index, columns=["CTNET"])

In [50]:
predicciones["CTNET"] = resultados["CTNET"]
predicciones

,Generación,LightGBM,Random Forest,CTNET
15550,0.0,0.0,0.0,NaN
15551,0.0,0.0,0.0,NaN
15552,0.0,0.0,0.0,NaN
15553,0.0,0.0,0.0,NaN
15554,0.0,0.0,0.0,NaN
...,...,...,...,...
18285,0.0,98.179402,0.0,0.000000
18286,0.0,0.0,0.0,29.706303
18287,0.0,0.0,0.0,76.294067
18288,0.0,4.376568,0.0,76.286552


In [51]:
predicciones["CTNET"] = predicciones["CTNET"].fillna(0)

In [52]:
# import optuna
# import tensorflow as tf
# from tensorflow.keras import layers
# from sklearn.model_selection import train_test_split

# # Definir la función objetivo para Optuna
# def objective(trial):
#     # Sugerir valores para los hiperparámetros
#     head_size = trial.suggest_int("head_size", 8, 64, step=8)
#     num_heads = trial.suggest_int("num_heads", 2, 8, step=2)
#     ff_dim = trial.suggest_int("ff_dim", 32, 256, step=32)
#     num_transformer_blocks = trial.suggest_int("num_transformer_blocks", 1, 4)
#     mlp_units = trial.suggest_categorical("mlp_units", [[128, 64], [256, 128, 64], [512, 256, 128]])
#     dropout = trial.suggest_float("dropout", 0.1, 0.5, step=0.1)
#     mlp_dropout = trial.suggest_float("mlp_dropout", 0.1, 0.5, step=0.1)
#     learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)

#     # Construcción del modelo con los hiperparámetros sugeridos
#     model = build_model(
#         input_shape=X_train_windowed.shape[1:],
#         head_size=head_size,
#         num_heads=num_heads,
#         ff_dim=ff_dim,
#         num_transformer_blocks=num_transformer_blocks,
#         mlp_units=mlp_units,
#         dropout=dropout,
#         mlp_dropout=mlp_dropout
#     )

#     # Compilar el modelo con los hiperparámetros sugeridos
#     model.compile(
#         loss=tf.keras.losses.MeanSquaredError(),
#         optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
#         metrics=[tf.keras.metrics.RootMeanSquaredError()]
#     )

#     # Entrenamiento con un número reducido de épocas para acelerar la búsqueda
#     history = model.fit(
#         X_train_windowed, y_train_windowed,
#         validation_split=0.2,
#         epochs=50,  # Reducimos las épocas para acelerar la búsqueda
#         batch_size=512,
#         verbose=0
#     )

#     # Obtener la métrica de validación (RMSE) y minimizarla
#     val_rmse = min(history.history["val_root_mean_squared_error"])
    
#     return val_rmse  # Queremos minimizar el RMSE

# # Ejecutar la optimización de hiperparámetros
# study = optuna.create_study(direction="minimize")
# study.optimize(objective, n_trials=20, timeout=3600)  # 20 iteraciones, máximo 1 hora

# # Mostrar los mejores hiperparámetros encontrados
# best_params = study.best_params
# print(f"Mejores hiperparámetros: {best_params}")


In [53]:
CTNET = build_model((X_train_windowed.shape[1], X_train_windowed.shape[2]), head_size=16, num_heads=8, ff_dim=256, num_transformer_blocks=1, mlp_units=[128,64], mlp_dropout=0.2, dropout=0.2)

In [54]:
history = compile_and_fit(CTNET, learning_rate = 0.0010762230908145116)

Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 434s 8s/step - loss: 0.1950 - mean_absolute_error: 0.3268 - mean_absolute_percentage_error: 65913784.0000 - root_mean_squared_error: 0.4410 - val_loss: 0.1035 - val_mean_absolute_error: 0.2901 - val_mean_absolute_percentage_error: 128536672.0000 - val_root_mean_squared_error: 0.3217
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 190s 7s/step - loss: 0.1364 - mean_absolute_error: 0.3277 - mean_absolute_percentage_error: 124453736.0000 - root_mean_squared_error: 0.3688 - val_loss: 0.0278 - val_mean_absolute_error: 0.1283 - val_mean_absolute_percentage_error: 44987248.0000 - val_root_mean_squared_error: 0.1668
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 208s 7s/step - loss: 0.0259 - mean_absolute_error: 0.1068 - mean_absolute_percentage_error: 16003262.0000 - root_mean_squared_error: 0.1608 - val_loss: 0.0180 - val_mean_absolute_error: 0.0862 - val_mean_absolute_percentage_error: 13951942.0000 - val_root_mean_squared_error: 0.1343
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━

In [55]:
CTNET_predictions = CTNET.predict(X_test_windowed)
CTNET_predictions

85/85 ━━━━━━━━━━━━━━━━━━━━ 24s 162ms/step


array([[0.00271126],
       [0.00256617],
       [0.0034628 ],
       ...,
       [0.00197012],
       [0.00247713],
       [0.00295433]], dtype=float32)

In [56]:
CTNET_predictions = y_scaler.inverse_transform(CTNET_predictions.reshape(-1, 1))
CTNET_predictions = np.clip(CTNET_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [57]:
resultados = pd.DataFrame(CTNET_predictions, index = y_test_windowed.index, columns=["CTNET"])

In [58]:
predicciones["CTNET"] = resultados["CTNET"]
predicciones

,Generación,LightGBM,Random Forest,CTNET
15550,0.0,0.0,0.0,NaN
15551,0.0,0.0,0.0,NaN
15552,0.0,0.0,0.0,NaN
15553,0.0,0.0,0.0,NaN
15554,0.0,0.0,0.0,NaN
...,...,...,...,...
18285,0.0,98.179402,0.0,98.742622
18286,0.0,0.0,0.0,36.379917
18287,0.0,0.0,0.0,59.103592
18288,0.0,4.376568,0.0,74.313957


## Forecasting

In [59]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import RootMeanSquaredError
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.losses import Huber
from tensorflow.keras.callbacks import EarlyStopping

In [60]:
Forecast_model = Sequential()
Forecast_model.add(InputLayer((X_train_windowed.shape[1], X_train_windowed.shape[2])))

#CNN
Forecast_model.add(Conv1D(filters=64, kernel_size=2, padding='same', activation='relu'))
Forecast_model.add(BatchNormalization())  # 🔹 Nueva Normalización aquí
Forecast_model.add(MaxPooling1D(pool_size=2))

#model_Soleado.add(Flatten())
#BiLSTM
Forecast_model.add(Bidirectional(LSTM(128, return_sequences=True)))
Forecast_model.add(Bidirectional(LSTM(64, return_sequences=True)))
Forecast_model.add(Dropout(0.2))  # 🔹 Mayor regularización en BiLSTM
Forecast_model.add(Bidirectional(LSTM(32, return_sequences=False)))

#Normalización y Dropout
Forecast_model.add(BatchNormalization())
Forecast_model.add(Dropout(0.3))

# Capas Densas
Forecast_model.add(Dense(16, activation='relu'))
Forecast_model.add(Dense(1, 'relu'))

Forecast_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_12 (Conv1D)              │ (None, 48, 64)         │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 24, 256)        │       197,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 24, 128)        │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 405,601 (1.55 MB)

 Trainable params: 405,345 (1.55 MB)

 Non-trainable params: 256 (1.00 KB)

In [61]:
cp = ModelCheckpoint('Forcasting_model.keras', save_best_only=True)
Forecast_model.compile(optimizer=Adam(learning_rate=0.0001), loss=Huber(delta=1000), metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [62]:
history = Forecast_model.fit(X_train_windowed, y_train_windowed, validation_data=(X_val_windowed, y_val_windowed), epochs=100, batch_size=8, callbacks=[cp, early_stop])

Epoch 1/100
1593/1593 ━━━━━━━━━━━━━━━━━━━━ 374s 197ms/step - loss: 0.0563 - mae: 0.1942 - val_loss: 0.0065 - val_mae: 0.0651
Epoch 2/100
1593/1593 ━━━━━━━━━━━━━━━━━━━━ 235s 141ms/step - loss: 0.0226 - mae: 0.1191 - val_loss: 0.0069 - val_mae: 0.0653
Epoch 3/100
1593/1593 ━━━━━━━━━━━━━━━━━━━━ 362s 202ms/step - loss: 0.0182 - mae: 0.1070 - val_loss: 0.0046 - val_mae: 0.0539
Epoch 4/100
1593/1593 ━━━━━━━━━━━━━━━━━━━━ 368s 192ms/step - loss: 0.0149 - mae: 0.0964 - val_loss: 0.0048 - val_mae: 0.0562
Epoch 5/100
1593/1593 ━━━━━━━━━━━━━━━━━━━━ 324s 192ms/step - loss: 0.0127 - mae: 0.0880 - val_loss: 0.0042 - val_mae: 0.0525
Epoch 6/100
1593/1593 ━━━━━━━━━━━━━━━━━━━━ 350s 209ms/step - loss: 0.0111 - mae: 0.0822 - val_loss: 0.0030 - val_mae: 0.0433
Epoch 7/100
1593/1593 ━━━━━━━━━━━━━━━━━━━━ 376s 203ms/step - loss: 0.0103 - mae: 0.0787 - val_loss: 0.0039 - val_mae: 0.0514
Epoch 8/100
1593/1593 ━━━━━━━━━━━━━━━━━━━━ 288s 180ms/step - loss: 0.0093 - mae: 0.0744 - val_loss: 0.0029 - val_mae: 0.0408


In [63]:
Forecast_predictions = Forecast_model.predict(X_test_windowed)
Forecast_predictions

85/85 ━━━━━━━━━━━━━━━━━━━━ 29s 192ms/step


array([[0.],
       [0.],
       [0.],
       ...,
       [0.],
       [0.],
       [0.]], dtype=float32)

In [64]:
Forecast_predictions = y_scaler.inverse_transform(Forecast_predictions.reshape(-1, 1))
Forecast_predictions = np.clip(Forecast_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [65]:
Forecast_resultados = pd.DataFrame(Forecast_predictions, index = y_test_windowed.index, columns=["Forecast"])

In [66]:
predicciones["Forecast"] = Forecast_resultados["Forecast"]
predicciones

,Generación,LightGBM,Random Forest,CTNET,Forecast
15550,0.0,0.0,0.0,NaN,NaN
15551,0.0,0.0,0.0,NaN,NaN
15552,0.0,0.0,0.0,NaN,NaN
15553,0.0,0.0,0.0,NaN,NaN
15554,0.0,0.0,0.0,NaN,NaN
...,...,...,...,...,...
18285,0.0,98.179402,0.0,98.742622,0.0
18286,0.0,0.0,0.0,36.379917,0.0
18287,0.0,0.0,0.0,59.103592,0.0
18288,0.0,4.376568,0.0,74.313957,0.0


## Métricas

In [67]:
predicciones.loc[~predicciones['CTNET'].isna(),'Generación']

15598    0.0
15599    0.0
15600    0.0
15601    0.0
15602    0.0
        ... 
18285    0.0
18286    0.0
18287    0.0
18288    0.0
18289    0.0
Name: Generación, Length: 2692, dtype: float64

## Photovoltaic

In [68]:
from tensorflow.keras.models import Model
inputs = Input(shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]))

# Primera capa CNN
x = Conv1D(filters=64, kernel_size=4, padding='same', activation='relu')(inputs)
x = MaxPooling1D(pool_size=2)(x)

# Segunda capa CNN
x = Conv1D(filters=128, kernel_size=4, padding='same', activation='relu')(x)
x = MaxPooling1D(pool_size=2)(x)

# Capa BiGRU
x = Bidirectional(GRU(64, return_sequences=True))(x)

# Atención: se define de forma explícita
attention = MultiHeadAttention(num_heads=4, key_dim=128)(x, x)

# Aplanar y agregar Dropout
x = Flatten()(attention)
x = Dropout(0.4)(x)
initializer = tf.keras.initializers.HeNormal()
x = Dense(64, activation="relu", kernel_regularizer=l2(0.01))(x)
x = Dense(32, activation="relu")(x)  # Otra capa intermedia

# Capa de salida
outputs = Dense(1, activation="linear")(x)

# Definir el modelo
Photo_model = Model(inputs=inputs, outputs=outputs)

# Resumen del modelo
Photo_model.summary()

Model: "functional_13"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 48, 6)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_13 (Conv1D)  │ (None, 48, 64)    │      1,600 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 24, 64)    │          0 │ conv1d_13[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_14 (Conv1D)  │ (None, 24, 128)   │     32,896 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 12, 128)   │          0 │ conv1d_14[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_3     │ (None, 12, 128)   │     74,496 │ max_pooling1d_2[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 12, 128)   │    263,808 │ bidirectional_3[… │
│ (MultiHeadAttentio… │                   │            │ bidirectional_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 1536)      │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 1536)      │          0 │ flatten[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 64)        │     98,368 │ dropout_11[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 32)        │      2,080 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 1)         │         33 │ dense_11[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 473,281 (1.81 MB)

 Trainable params: 473,281 (1.81 MB)

 Non-trainable params: 0 (0.00 B)

In [69]:
cp2 = ModelCheckpoint('Photovoltaic_model.keras', save_best_only=True)
Photo_model.compile(optimizer=Adam(learning_rate=0.0001), loss="mean_squared_error", metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [70]:
history = Photo_model.fit(
    X_train_windowed, y_train_windowed,
    validation_data=(X_val_windowed, y_val_windowed),
    epochs=50,
    batch_size=16,
    callbacks=[cp, early_stop]
)

Epoch 1/50
797/797 ━━━━━━━━━━━━━━━━━━━━ 107s 76ms/step - loss: 0.7560 - mae: 0.2294 - val_loss: 0.0513 - val_mae: 0.0573
Epoch 2/50
797/797 ━━━━━━━━━━━━━━━━━━━━ 75s 64ms/step - loss: 0.0384 - mae: 0.0666 - val_loss: 0.0107 - val_mae: 0.0471
Epoch 3/50
797/797 ━━━━━━━━━━━━━━━━━━━━ 73s 53ms/step - loss: 0.0150 - mae: 0.0594 - val_loss: 0.0069 - val_mae: 0.0429
Epoch 4/50
797/797 ━━━━━━━━━━━━━━━━━━━━ 93s 65ms/step - loss: 0.0123 - mae: 0.0563 - val_loss: 0.0069 - val_mae: 0.0450
Epoch 5/50
797/797 ━━━━━━━━━━━━━━━━━━━━ 83s 65ms/step - loss: 0.0103 - mae: 0.0517 - val_loss: 0.0063 - val_mae: 0.0422
Epoch 6/50
797/797 ━━━━━━━━━━━━━━━━━━━━ 81s 61ms/step - loss: 0.0106 - mae: 0.0521 - val_loss: 0.0085 - val_mae: 0.0550
Epoch 7/50
797/797 ━━━━━━━━━━━━━━━━━━━━ 84s 63ms/step - loss: 0.0097 - mae: 0.0509 - val_loss: 0.0049 - val_mae: 0.0370
Epoch 8/50
797/797 ━━━━━━━━━━━━━━━━━━━━ 83s 63ms/step - loss: 0.0095 - mae: 0.0500 - val_loss: 0.0058 - val_mae: 0.0419
Epoch 9/50
797/797 ━━━━━━━━━━━━━━━━━━━━

In [71]:
Photo_predictions = Photo_model.predict(X_test_windowed)
Photo_predictions

85/85 ━━━━━━━━━━━━━━━━━━━━ 6s 64ms/step


array([[ 4.0663043e-03],
       [ 2.9067779e-03],
       [ 8.9708809e-04],
       ...,
       [ 1.6839420e-03],
       [ 9.8706223e-05],
       [-2.2835564e-04]], dtype=float32)

In [72]:
Photo_predictions = y_scaler.inverse_transform(Photo_predictions.reshape(-1, 1))
Photo_predictions = np.clip(Photo_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [73]:
Photo_resultados = pd.DataFrame(Photo_predictions, index = y_test_windowed.index, columns=["Photo"])

In [74]:
predicciones["Photo"] = Photo_resultados["Photo"]
predicciones

,Generación,LightGBM,Random Forest,CTNET,Forecast,Photo
15550,0.0,0.0,0.0,NaN,NaN,NaN
15551,0.0,0.0,0.0,NaN,NaN,NaN
15552,0.0,0.0,0.0,NaN,NaN,NaN
15553,0.0,0.0,0.0,NaN,NaN,NaN
15554,0.0,0.0,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...
18285,0.0,98.179402,0.0,98.742622,0.0,104.966263
18286,0.0,0.0,0.0,36.379917,0.0,83.929703
18287,0.0,0.0,0.0,59.103592,0.0,50.518261
18288,0.0,4.376568,0.0,74.313957,0.0,2.961187


In [75]:
print("LightGBM")
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['LightGBM'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print("Random Forest")
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['Random Forest']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['Random Forest'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['Random Forest']):.4f}")
print("CTNET")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET']):.4f}")
print("Forecast")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast']):.4f}")
print("Photovoltaic")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo']):.4f}")

LightGBM
MAE: 926.3719
RMSE: 2052.1413
R²: 0.9686
Random Forest
MAE: 963.5648
RMSE: 2319.0766
R²: 0.9600
CTNET
MAE: 1338.6782
RMSE: 2786.9153
R²: 0.9419
Forecast
MAE: 1361.2422
RMSE: 2806.0739
R²: 0.9411
Photovoltaic
MAE: 1273.9848
RMSE: 2673.2517
R²: 0.9465


In [76]:
# Seleccionar las columnas desde "LightGBM" en adelante
columnas_nuevas = predicciones.loc[:, "LightGBM":]

# Unir con `datos` usando el índice, manteniendo todo en `datos`
datos = datos.merge(columnas_nuevas, left_index=True, right_index=True, how='left')

# Ver resultado
datos.head()


,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day,LightGBM,Random Forest,CTNET,Forecast,Photo
24,2022-09-02 00:00:00,0.0,19,76,0,0,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
25,2022-09-02 01:00:00,0.0,18,81,0,1,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
26,2022-09-02 02:00:00,0.0,18,84,0,2,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
27,2022-09-02 03:00:00,0.0,18,86,0,3,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
28,2022-09-02 04:00:00,0.0,17,86,0,4,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN


## X_train para hacer análisis de sobreajuste

In [77]:
predicciones_train = y_train.copy()

In [78]:
LightGBM_predictions_train = LightGBM_model.predict(X_train_scaled_df)
LightGBM_predictions_train = y_scaler.inverse_transform(LightGBM_predictions_train.reshape(-1, 1))
LightGBM_predictions_train = np.clip(LightGBM_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
LightGBM_resultados = pd.DataFrame(LightGBM_predictions_train, index = y_train_scaled_df.index, columns=["LightGBM_train"])
predicciones_train["LightGBM_train"] = LightGBM_resultados["LightGBM_train"]

[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85


In [79]:
RandomForest_predictions_train = RF_model.predict(X_train_scaled_df)
RandomForest_predictions_train = y_scaler.inverse_transform(RandomForest_predictions_train.reshape(-1, 1))
RandomForest_predictions_train = np.clip(RandomForest_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
RandomForest_resultados = pd.DataFrame(RandomForest_predictions_train, index = y_train_scaled_df.index, columns=["RandomForest_train"])
predicciones_train["RandomForest_train"] = RandomForest_resultados["RandomForest_train"]

In [80]:
CTNET_predictions_train = CTNET.predict(X_train_windowed)
CTNET_predictions_train = y_scaler.inverse_transform(CTNET_predictions_train.reshape(-1, 1))
CTNET_predictions_train = np.clip(CTNET_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
CTNET_resultados = pd.DataFrame(CTNET_predictions_train, index = y_train_windowed.index, columns=["CTNET_train"])
predicciones_train["CTNET_train"] = CTNET_resultados["CTNET_train"]

 24/399 ━━━━━━━━━━━━━━━━━━━━ 13s 36ms/step

399/399 ━━━━━━━━━━━━━━━━━━━━ 16s 39ms/step


In [81]:
Forecast_predictions_train = Forecast_model.predict(X_train_windowed)
Forecast_predictions_train = y_scaler.inverse_transform(Forecast_predictions_train.reshape(-1, 1))
Forecast_predictions_train = np.clip(Forecast_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
Forecast_resultados = pd.DataFrame(Forecast_predictions_train, index = y_train_windowed.index, columns=["Forecast_train"])
predicciones_train["Forecast_train"] = Forecast_resultados["Forecast_train"]

399/399 ━━━━━━━━━━━━━━━━━━━━ 21s 49ms/step


In [82]:
Photo_predictions_train = Photo_model.predict(X_train_windowed)
Photo_predictions_train = y_scaler.inverse_transform(Photo_predictions_train.reshape(-1, 1))
Photo_predictions_train = np.clip(Photo_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
Photo_resultados = pd.DataFrame(Photo_predictions_train, index = y_train_windowed.index, columns=["Photo_train"])
predicciones_train["Photo_train"] = Photo_resultados["Photo_train"]

399/399 ━━━━━━━━━━━━━━━━━━━━ 9s 21ms/step


In [83]:
predicciones_train

,Generación,LightGBM_train,RandomForest_train,CTNET_train,Forecast_train,Photo_train
24,0.0,0.828967,0.000000,NaN,NaN,NaN
25,0.0,0.000000,0.000000,NaN,NaN,NaN
26,0.0,4.369784,0.000000,NaN,NaN,NaN
27,0.0,1.320306,0.000000,NaN,NaN,NaN
28,0.0,1.673732,0.000000,NaN,NaN,NaN
...,...,...,...,...,...,...
12805,24870.0,23491.893596,24325.388118,21691.634766,17166.798828,20089.230469
12806,21574.0,22315.919455,21670.986854,20648.957031,18124.027344,20865.185547
12807,21167.0,18420.663022,21333.927262,19735.882812,17181.228516,19843.593750
12808,23845.0,18420.979124,22321.621978,16522.562500,17705.730469,19769.056641


In [84]:
print("LightGBM")
print(f"MAE: {mean_absolute_error(predicciones_train['Generación'], predicciones_train['LightGBM_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train['Generación'], predicciones_train['LightGBM_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train['Generación'], predicciones_train['LightGBM_train']):.4f}")
print("Random Forest")
print(f"MAE: {mean_absolute_error(predicciones_train['Generación'], predicciones_train['RandomForest_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train['Generación'], predicciones_train['RandomForest_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train['Generación'], predicciones_train['RandomForest_train']):.4f}")
print("CTNET")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train']):.4f}")
print("Forecast")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train']):.4f}")
print("Photovoltaic")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train']):.4f}")

LightGBM
MAE: 721.0879
RMSE: 1612.1754
R²: 0.9793
Random Forest
MAE: 286.2313
RMSE: 701.9581
R²: 0.9961
CTNET
MAE: 1120.4497
RMSE: 2449.2552
R²: 0.9522
Forecast
MAE: 1332.4104
RMSE: 2772.9398
R²: 0.9387
Photovoltaic
MAE: 1273.8304
RMSE: 2650.3934
R²: 0.9440


In [85]:
# Seleccionar las columnas desde "LightGBM" en adelante
columnas_nuevas = predicciones_train.loc[:, "LightGBM_train":]

# Unir con `datos` usando el índice, manteniendo todo en `datos`
datos = datos.merge(columnas_nuevas, left_index=True, right_index=True, how='left')

# Ver resultado
datos.head()


,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day,LightGBM,Random Forest,CTNET,Forecast,Photo,LightGBM_train,RandomForest_train,CTNET_train,Forecast_train,Photo_train
24,2022-09-02 00:00:00,0.0,19,76,0,0,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.828967,0.0,NaN,NaN,NaN
25,2022-09-02 01:00:00,0.0,18,81,0,1,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.000000,0.0,NaN,NaN,NaN
26,2022-09-02 02:00:00,0.0,18,84,0,2,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,4.369784,0.0,NaN,NaN,NaN
27,2022-09-02 03:00:00,0.0,18,86,0,3,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,1.320306,0.0,NaN,NaN,NaN
28,2022-09-02 04:00:00,0.0,17,86,0,4,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN,1.673732,0.0,NaN,NaN,NaN


In [86]:
datos.to_excel("04.7_Predicciones_Conjunto_completo CTNET.xlsx", index=True)

## Guardamos los modelos

In [87]:
import joblib

# Guardar modelo LightGBM
joblib.dump(LightGBM_model, "4_7_LightGBM_model.pkl")

# Guardar modelo Random Forest
joblib.dump(RF_model, "4_7_RandomForest_model.pkl")


['4_7_RandomForest_model.pkl']

In [88]:
CTNET.save("4_7_CTNET_model.keras")
Forecast_model.save("4_7_Forecast_model.keras")
Photo_model.save("4_7_Photo_model.keras")